# 07. 나만의 분류기 — 특징 뽑기 + numpy 로 직접 학습

> 큰 모델을 처음부터 학습하려면 사진 수십만 장과 GPU 며칠이 든다. 대신 **이미 배운 모델의 눈**을 빌린다(전이 학습).
> 그 눈이 사진을 숫자 1280개로 요약해 주면, 그 위에 **아주 작은 모델**만 우리 사진 수십 장으로 학습한다.
> edge-lab 의 [가르치기] 가 하는 일과 같다 (거기는 브라우저에서 학습한다).

### 이 시간에 배우는 것
1. **특징 추출** — MobileNetV2 의 마지막 판정층 바로 앞을 꺼내 쓰기
2. **소프트맥스 회귀** 를 수식부터 numpy 코드까지 직접 구현 (경사 하강)
3. 학습 / 시험 나누기, 정확도, **혼동 행렬**, 과적합
4. 더 간단한 방법(가장 가까운 평균)과 비교

### 전체 흐름
```
사진 ─▶ [MobileNetV2 (고정, NPU)] ─▶ 숫자 1280개 ─▶ [소프트맥스 회귀 W(1280×K), b(K)] ─▶ K 가지 확률
          이미 배운 눈 (안 바꾼다)       특징              우리가 학습하는 부분 (numpy)
```

## 1. 특징 뽑는 모델
`mobilenetv2_feat` 는 MobileNetV2 에서 "1000가지 중 무엇인가" 를 정하는 마지막 층을 떼어 낸 것이다.
그래서 출력이 정답이 아니라 **사진을 요약한 숫자 1280개**다. 비슷한 사진은 비슷한 숫자가 나온다.
입력은 `(1, 224, 224, 3)` — **NHWC**, RGB, 0~1 (가르치기 화면의 TF.js 와 같은 전처리).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame


DEVICE = pick("NPU", "GPU", "CPU")
backbone = core.compile_model(MODELS / "backbone" / "mobilenetv2_feat.xml", DEVICE)
print(DEVICE, "입력:", backbone.input(0).shape, "출력:", backbone.output(0).shape)

def feature(bgr):
    """사진 → 숫자 1280개."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255.0
    return backbone(x[None])[backbone.output(0)].reshape(-1)

## 2. 사진 모으기
`LABELS` 에 종류를 적는다. 종류마다 웹캠이 `N` 장을 조금씩 간격을 두고 찍는다 — **그동안 물건의 각도 · 거리 · 위치를 바꿔 준다**.
다양하게 찍을수록 새 사진에도 잘 맞는다. 웹캠이 없으면 `data/<종류>/*.jpg` 폴더에서 읽는다.

`input()` 은 노트북 안에 입력 칸을 띄운다 — Enter 를 누르면 찍기 시작한다.

In [ ]:
LABELS = ["가위", "바위", "보"]
N = 20

def capture(n, index=0, gap=0.15):
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    shots = []
    for _ in range(5):
        cap.read()                                    # 처음 몇 장은 버린다
    while len(shots) < n:
        ok, f = cap.read()
        if not ok:
            break
        shots.append(f)
        time.sleep(gap)
    cap.release()
    return shots

X, y = [], []
for k, label in enumerate(LABELS):
    folder = Path("data") / label
    if folder.is_dir():
        shots = [cv2.imread(str(p)) for p in sorted(folder.glob("*.jpg"))]
    else:
        input(f"'{label}' 을(를) 웹캠에 보여 주고 Enter")
        shots = capture(N)
    shots = [s for s in shots if s is not None]
    print(f"{label}: {len(shots)}장")
    for s in shots:
        X.append(feature(s))
        y.append(k)
X, y = np.array(X, np.float32), np.array(y)
print("특징 행렬 X:", X.shape, "| 정답 y:", y.shape)

### 특징을 눈으로 — 1280 차원을 2 차원으로 눌러 보기 (PCA)
1280개 숫자는 그릴 수 없으니, 가장 많이 퍼진 두 방향으로 눌러서 점으로 찍는다.
종류별 점들이 **따로 뭉쳐 있으면** 쉽게 나눌 수 있다는 뜻이다.

In [ ]:
import matplotlib.pyplot as plt     # 그래프 글자는 영어로 (한글은 글꼴 설정이 따로 필요)
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)    # PCA = 중심을 뺀 행렬의 SVD
P = Xc @ Vt[:2].T                                    # 두 방향으로 투영
plt.figure(figsize=(4.5, 4))
for k in range(len(LABELS)):
    plt.scatter(P[y == k, 0], P[y == k, 1], s=18, label=f"class {k}")
plt.legend(); plt.title("features (PCA 2D)"); plt.show()
print("종류 번호:", {k: l for k, l in enumerate(LABELS)})

## 3. 소프트맥스 회귀 — 수식에서 코드로

**점수**: 종류마다 점수 하나. $s = x W + b$ ($x$: 특징 1280개, $W$: 1280 × K, $b$: K개)

**확률**(소프트맥스): 점수를 0~1, 합 1 로 바꾼다. $p_k = \dfrac{e^{s_k}}{\sum_j e^{s_j}}$

**손실**(교차 엔트로피): 정답 종류의 확률이 1 에 가까울수록 작다. $L = -\log p_{\text{정답}}$

**기울기**: 놀랍게도 간단하다. $\dfrac{\partial L}{\partial s} = p - \text{onehot}(\text{정답})$ → $\dfrac{\partial L}{\partial W} = x^\top (p - \text{onehot})$

**경사 하강**: $W \leftarrow W - \eta \cdot \text{기울기}$ ($\eta$: 학습률 `lr`). 조금씩 손실이 줄어드는 쪽으로 고친다.

**규제**(`lam`): $W$ 가 너무 커지지 않게 조금씩 줄인다 — 학습 사진에만 맞추는 **과적합**을 줄인다.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
cut = int(len(X) * 0.8)                               # 80% 로 배우고 20% 로 시험
tr, te = idx[:cut], idx[cut:]

mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6           # 특징마다 크기를 맞춘다 (학습 사진의 평균 · 표준편차로만!)
Z = (X - mu) / sd
K, D = len(LABELS), X.shape[1]
W = np.zeros((D, K), np.float32)
b = np.zeros(K, np.float32)

def softmax(s):
    s = s - s.max(1, keepdims=True)                   # 큰 수 넘침 방지 (결과는 같다)
    e = np.exp(s)
    return e / e.sum(1, keepdims=True)

onehot = np.eye(K)[y]
lr, lam = 0.05, 1e-3
history = []
for step in range(301):
    p = softmax(Z[tr] @ W + b)
    loss = float(-np.log(np.clip(p[np.arange(len(tr)), y[tr]], 1e-9, 1)).mean()) + 0.0   # + 0.0: -0.0 을 0.0 으로
    g = (p - onehot[tr]) / len(tr)                    # 점수에 대한 기울기
    W -= lr * (Z[tr].T @ g + lam * W)
    b -= lr * g.sum(0)
    history.append(loss)
    if step % 50 == 0:
        acc_tr = (p.argmax(1) == y[tr]).mean()
        acc_te = (softmax(Z[te] @ W + b).argmax(1) == y[te]).mean() if len(te) else float("nan")
        print(f"{step:3d}  손실 {loss:.3f}  배운 것 {acc_tr:.0%}  시험 {acc_te:.0%}")

plt.figure(figsize=(4.5, 2.4)); plt.plot(history); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

**읽는 법** — "배운 것" 은 학습에 쓴 사진의 정확도, "시험" 은 **처음 보는** 사진의 정확도. 실력은 시험으로 말한다.
배운 것은 100% 인데 시험이 낮으면 **과적합**: 사진을 더 다양하게 모으거나 `lam` 을 키운다.

## 4. 혼동 행렬 — 무엇을 무엇으로 헷갈리나
줄 = 정답, 칸 = 모델의 답. 대각선이 맞힌 것, 나머지가 헷갈린 것.

In [ ]:
pred_te = softmax(Z[te] @ W + b).argmax(1)
cm = np.zeros((K, K), int)
for t_, p_ in zip(y[te], pred_te):
    cm[t_, p_] += 1
print("정답 \\ 답 ", "  ".join(f"{l:>4s}" for l in LABELS))
for k in range(K):
    print(f"{LABELS[k]:>8s}  ", "  ".join(f"{v:4d}" for v in cm[k]))

## 5. 더 간단한 방법 — 가장 가까운 평균
종류마다 특징의 **평균**을 내 두고, 새 사진은 가장 가까운 평균의 종류로 정한다. 학습이랄 것도 없이 몇 줄이다.
사진이 적을 때는 이것도 꽤 잘 맞는다. 비교해 보자.

In [ ]:
centroids = np.stack([Z[tr][y[tr] == k].mean(0) for k in range(K)])          # 종류별 평균 (K × 1280)
d = ((Z[te][:, None, :] - centroids[None]) ** 2).sum(-1)                       # 시험 사진 × 종류 거리
acc_nc = (d.argmin(1) == y[te]).mean() if len(te) else float("nan")
acc_sm = (pred_te == y[te]).mean() if len(te) else float("nan")
print(f"시험 정확도 — 소프트맥스 회귀 {acc_sm:.0%} · 가장 가까운 평균 {acc_nc:.0%}")

## 6. 새 사진으로 맞혀 보기

In [ ]:
def predict(bgr):
    p = softmax(((feature(bgr) - mu) / sd)[None] @ W + b)[0]
    return LABELS[int(p.argmax())], p

shot = grab()
label, p = predict(shot)
show(shot, 320)
print("답:", label, "| 확률:", {l: round(float(v), 2) for l, v in zip(LABELS, p)})

## 7. 저장 · 불러오기
학습한 것은 `W`, `b` 와 정규화에 쓴 `mu`, `sd` 뿐이다 — 이것만 저장하면 된다 (특징 모델은 그대로 쓰니까).

In [ ]:
np.savez("my_classifier.npz", W=W, b=b, mu=mu, sd=sd, labels=np.array(LABELS))
m = np.load("my_classifier.npz")
print("저장한 종류:", list(m["labels"]), "| W", m["W"].shape, "| 파일 크기", round(Path("my_classifier.npz").stat().st_size / 1024), "KB")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 손실이 `nan` | `lr` 이 너무 크다 → 0.01 로 |
| 시험 정확도만 낮다 | 과적합 · 사진이 비슷비슷 → 다양하게 더 찍기, `lam` 키우기 |
| 새 사진에서 늘 같은 답 | 배경이 종류마다 달랐다 → 모델이 물건 대신 배경을 배웠다. 같은 배경에서 찍는다 |
| 시험 정확도가 `nan` | 사진이 너무 적어 시험 몫이 0장 |

## 정리
- 전이 학습: 큰 모델은 **고정된 눈**(특징 추출), 우리는 그 위의 **작은 판정층**만 학습한다.
- 소프트맥스 회귀 = 점수(xW+b) → 확률(softmax) → 손실(교차 엔트로피) → 기울기(p − 정답) → 경사 하강.
- 실력은 처음 보는 사진(시험)으로 잰다. 혼동 행렬로 무엇을 헷갈리는지 본다.

## 해 볼 것
1. `N` 을 5 로 줄이면 시험 정확도가 어떻게 되나? 40 으로 늘리면?
2. `lr` 을 0.5 · 0.005 로 바꾸면 손실 그래프가 어떻게 달라지나?
3. 종류를 하나 더 추가해 보자 (예: "아무것도 없음"). 이것이 왜 쓸모 있을까?
   <details><summary>힌트</summary>

   종류가 셋뿐이면 아무것도 안 보여 줘도 셋 중 하나로 답한다. "없음" 종류가 있으면 그것을 고를 수 있다.
   </details>
4. 06 번의 손 점 63개를 특징으로 바꿔 손동작 분류기를 만들어 보자. 배경이 바뀌어도 잘 맞히나?

---
# 🛠 미니 프로젝트 — 실시간 판정기 — "모르겠음" 까지

**만들 것**: 저장한 분류기(`my_classifier.npz`)를 **다시 불러와** 웹캠 영상에서 실시간으로 판정한다.
가장 큰 확률이 기준(`MIN_P`)보다 낮으면 억지로 고르지 않고 **"?"** 라고 한다. 판정이 바뀔 때마다 기록한다.

**쓰는 것**: `feature` · `softmax` (이 노트북), `np.load`, `live`

**왜 "모르겠음" 이 필요한가** — 소프트맥스는 늘 셋 중 하나에 확률을 몰아준다. 처음 보는 물건을 보여 줘도 "가위 0.5" 처럼 답한다.
확률이 낮으면 버리는 것만으로 엉뚱한 답이 많이 줄어든다.

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
m = np.load("my_classifier.npz")                      # 저장한 것을 다시 불러온다 (새로 학습하지 않는다)
W_, b_, mu_, sd_, labels_ = m["W"], m["b"], m["mu"], m["sd"], [str(l) for l in m["labels"]]
MIN_P = 0.6
changes, prev = [], [None]

def judge(f):
    p = softmax(((feature(f) - mu_) / sd_)[None] @ W_ + b_)[0]
    k = int(p.argmax())
    tag = f"class {k}" if p[k] >= MIN_P else "?"       # 사진에는 영어로 (한글은 cv2 가 못 쓴다)
    if tag != prev[0]:
        changes.append((time.strftime("%H:%M:%S"), labels_[k] if p[k] >= MIN_P else "모르겠음", round(float(p[k]), 2)))
        prev[0] = tag
    color = (0, 200, 0) if tag != "?" else (0, 0, 255)
    cv2.putText(f, f"{tag}  {p[k]:.2f}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    for i, v in enumerate(p):                          # 확률 막대
        cv2.rectangle(f, (10, 60 + i * 22), (10 + int(v * 200), 78 + i * 22), (255, 160, 0), -1)
    return f

live(judge, seconds=15)
print("종류 번호:", {f"class {i}": l for i, l in enumerate(labels_)})
print("판정이 바뀐 때:", changes)

**더 해 보기**
- 같은 판정이 1초 이상 이어질 때만 "확정" 으로 치기 (깜빡임 막기)
- 확정될 때 11 번처럼 보드로 명령 보내기
- `MIN_P` 를 0.4 · 0.8 로 바꿔 보고, "?" 가 얼마나 자주 나오는지 세어 보기